# Figure S4b–c — program CRT and replicate effect sizes

This is a re-scripted workflow, not a notebook launcher. Source Q-Q and replicate-effect functions reside in `figure_s04_panels.py`; the original embedded CRT backend and replicate computation are ordinary repository Python modules.

The default reads numerical Q-Q/p-value and replicate beta results and redraws every panel. `RECOMPUTE_CRT=True` recalculates those data from the original inputs (B=10000; B_REP=127) using programDE. It does not run or require any old notebook, and never copies previously rendered figures.

Source: `iPSC_EC_QQ_full_covariates_with_replicate_correlation.ipynb`, original source SHA-256 `6e80625abd9b5e71d45da6dd19e888898318d39a159a7ac35600fc1d85489e87`. Parameters, NTC exclusions, pairing, correlation, and plotnine styling are ported.

In [ ]:
from pathlib import Path
import sys, os, json
from IPython.display import display
REPO_ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "src" / "cc_perturbseq").is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Open this notebook inside the repository checkout.")
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))
from cc_perturbseq.figure_notebook import initialize_figure, load_figure_manifest, get_path, panel_directory, write_receipt

import pandas as pd
from cc_perturbseq.figure_s04_panels import *
from cc_perturbseq import crt_qq_backend as qqb
from cc_perturbseq.figure_s04_compute import compute_replicate_betas
ctx = initialize_figure(load_figure_manifest("Figure_S04"), require_inputs=False)
PANEL_B_DIR, PANEL_C_DIR = panel_directory(ctx,"b"), panel_directory(ctx,"c")
# Replot from numerical results, never from old figure files. Enable raw CRT
# recomputation explicitly when the per-day results/beta matrices need rebuilding.
RECOMPUTE_CRT = False
DAYS = ["d0","d1","d2","d3"]
K, DENSITY_THRESHOLD, B = 50, "0_2", 10000
NTC_LABELS = ["SAFE","non-targeting","NTC"]
N_JOBS = int(os.environ.get("SLURM_CPUS_PER_TASK","4"))
INPUT_MODEL_ROOT = Path(os.environ.get("CCPERTURBSEQ_S4_RESULTS", "/oak/stanford/groups/engreitz/Users/tri/CodeBase/programDE/notebooks/iPSC_EC_QQ_full_covariates_with_replicate_correlation_bundle/results/ipsc_ec_qq_exports_full_covariates/k_50.dt_0_2"))
PROGRAMDE_ROOT = Path(qqb.DEFAULT_PROGRAMDE_ROOT)
DATA_DIR, CNMF_DIR = Path(qqb.DEFAULT_DATA_DIR), Path(qqb.DEFAULT_CNMF_DIR)
GUIDE_LONG_TEMPLATE = qqb.DEFAULT_GUIDE_LONG_TEMPLATE
GUIDE_METADATA_TSV = get_path(ctx["paths"],"ccperturbseq.target_metadata")
RECOMPUTED_ROOT = ctx["output_dir"] / "recomputed_crt"
print("Numerical inputs:",INPUT_MODEL_ROOT)
print("Recompute raw CRT:",RECOMPUTE_CRT)


## Read or explicitly recompute the numerical inputs

In [ ]:
if RECOMPUTE_CRT:
    argv = ["--programde-root",str(PROGRAMDE_ROOT),"--data-dir",str(DATA_DIR),
        "--cnmf-dir",str(CNMF_DIR),"--guide-long-template",GUIDE_LONG_TEMPLATE,
        "--guide-metadata-tsv",str(GUIDE_METADATA_TSV),"--out-dir",str(RECOMPUTED_ROOT),
        "--days",*DAYS,"--k",str(K),"--density-threshold",DENSITY_THRESHOLD,
        "--B",str(B),"--n-jobs",str(N_JOBS),"--require-covariates","--overwrite","--no-plot"]
    for label in NTC_LABELS:
        argv += ["--ntc-label",label]
    qqb.main(argv)
    compute_replicate_betas(programde_root=PROGRAMDE_ROOT,data_dir=DATA_DIR,
        cnmf_dir=CNMF_DIR,guide_long_template=GUIDE_LONG_TEMPLATE,
        guide_metadata_tsv=GUIDE_METADATA_TSV,output_dir=RECOMPUTED_ROOT,
        days=DAYS,n_jobs=N_JOBS)
    INPUT_MODEL_ROOT = RECOMPUTED_ROOT / qqb.model_tag(K,DENSITY_THRESHOLD)

qq_csv = INPUT_MODEL_ROOT / "all_days_QQ_points.csv"
manifest_path = INPUT_MODEL_ROOT / "biological_replicate_effects/replicate_beta_manifest.csv"
for path in (qq_csv, manifest_path):
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(f"Missing numerical S4 input: {path}. Set INPUT_MODEL_ROOT to the results directory or enable RECOMPUTE_CRT. No previous notebook/figure is needed.")
qq_points = pd.read_csv(qq_csv)
# Recalculate Q-Q coordinates using the original backend rank formula.
if "pvalue" in qq_points:
    grouping = [c for c in ["day","pvalue_type","series_type"] if c in qq_points]
    qq_points = pd.concat([qqb.add_qq_columns(g.copy()) for _,g in qq_points.groupby(grouping,sort=False)],ignore_index=True)
qq_points.to_csv(PANEL_B_DIR / "all_days_QQ_points.csv",index=False)
manifest = pd.read_csv(manifest_path)
manifest = manifest.loc[manifest["status"].isin(["written","existing"]) & manifest["beta_path"].notna()].copy()
# Only explicit relative paths or the exact source basename in its day folder
# are relocated; never scan arbitrary beta tables or copy finished figures.
for index,row in manifest.iterrows():
    recorded = Path(str(row["beta_path"]))
    candidates = [recorded] if recorded.is_absolute() else [manifest_path.parent / recorded, INPUT_MODEL_ROOT / recorded]
    candidates += [manifest_path.parent / "betas_by_replicate" / str(row["day"]) / recorded.name]
    found = next((p for p in candidates if p.is_file() and p.stat().st_size),None)
    if found is None:
        raise FileNotFoundError(f"Missing beta matrix: {recorded}; checked {candidates}")
    manifest.loc[index,"beta_path"] = str(found.resolve())
if manifest.empty:
    raise ValueError("No usable numerical beta matrices in replicate manifest.")
display(manifest)


## S4b — recompute empirical Q-Q coordinates and render per-day panels

In [ ]:

illustrator_out_dir = PANEL_B_DIR

all_saved_files = {}

for day in DAYS:
    p, saved = plot_ipsc_ec_qq(
        qq_points,
        day=day,
        pvalue_type="empirical",
        title=f"{day}: Real target and NTC empirical p-values",
        subtitle="RNA QC, guide count and guide UMI burden",
        output_stem=illustrator_out_dir / f"{day}_QQ_full_covariates",
        formats=("pdf", "svg", "eps", "png"),
    )
    all_saved_files[day] = saved
    display(p)

print("Saved files:")
for day, files in all_saved_files.items():
    print(f"\n{day}")
    for fmt, path in files.items():
        print(f"  {fmt.upper()}: {path}")


## S4c — match biological-replicate beta estimates and render correlations

In [ ]:
PLOT_OUT_DIR = PANEL_C_DIR
# Build and save the matched effect table.
effects = build_effect_pair_table(manifest)
summary = summarize_effect_pairs(effects)
effects = effects.merge(
    summary[["day", "replicate_pair", "facet_label"]],
    on=["day", "replicate_pair"],
    how="left",
)

summary_path = PLOT_OUT_DIR / "biological_replicate_DE_program_effect_correlation_summary.csv"
effects_path = PLOT_OUT_DIR / "biological_replicate_DE_program_effect_pairs.csv"
summary.to_csv(summary_path, index=False)
effects.to_csv(effects_path, index=False)

display(summary)

# Combined plot across all days and all replicate pairs.
combined_plot, combined_width, combined_height = plot_effect_correlation(
    effects,
    title="Biological replicate correlation of DE program effects",
    subtitle="Each point is one real target × cNMF program beta effect",
    ncol=2,
)
display(combined_plot)

combined_stem = PLOT_OUT_DIR / "biological_replicate_DE_program_effect_correlation_all_days"
combined_saved = save_plotnine_formats(
    combined_plot,
    combined_stem,
    width=combined_width,
    height=combined_height,
)

# Also save one faceted plot per day.
per_day_saved = {}
for day in DAYS:
    day_effects = effects[effects["day"].astype(str).eq(str(day))].copy()
    if day_effects.empty:
        continue

    n_facets = int(day_effects["facet_label"].nunique())
    ncol = 1 if n_facets == 1 else 2
    day_plot, day_width, day_height = plot_effect_correlation(
        day_effects,
        title=f"{day}: biological replicate correlation of DE program effects",
        subtitle="Each point is one real target × cNMF program beta effect",
        ncol=ncol,
    )
    display(day_plot)

    day_stem = PLOT_OUT_DIR / f"{day}_biological_replicate_DE_program_effect_correlation"
    per_day_saved[day] = save_plotnine_formats(
        day_plot,
        day_stem,
        width=day_width,
        height=day_height,
    )

print("Saved data tables:")
print("  ", summary_path)
print("  ", effects_path)

print("\nSaved combined plot files:")
for fmt, path in combined_saved.items():
    print(f"  {fmt.upper()}: {path}")

print("\nSaved per-day plot files:")
for day, saved in per_day_saved.items():
    print(day)
    for fmt, path in saved.items():
        print(f"  {fmt.upper()}: {path}")


In [ ]:
for panel,directory in [("b",PANEL_B_DIR),("c",PANEL_C_DIR)]:
    files = sorted(p for p in directory.glob("*") if p.is_file())
    if not any(p.suffix == ".svg" for p in files):
        raise RuntimeError(f"Panel {panel}: SVG export failed; inspect the plotting error.")
    write_receipt(ctx,panel,status="RESCRIPTED_NUMERICAL_INPUTS_RENDERED",
        inputs={"qq_points":str(qq_csv),"replicate_manifest":str(manifest_path)},
        outputs={p.name:str(p) for p in files},
        parameters={"runtime_source_notebook_required":False,"recomputed_crt":RECOMPUTE_CRT,
                    "source_style_changed":False,"panels":["b","c"]})
